In [1]:
import torch
import numpy as np
import pandas as pd
import transformers
import sys
import os
src_path = os.path.abspath(os.path.join(os.getcwd(), '..', ''))
sys.path.append(src_path)
from transformers import BertTokenizer
# evaluation
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
import torch
from torch.utils.data import Dataset
import re

class AutomaticScoringDataset(Dataset):
    def __init__(self, dataframe, tokenizer, use_reference=False):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.use_reference = use_reference  # Menentukan apakah menggunakan reference answer atau tidak

    def __len__(self):
        return len(self.dataframe)

    def preprocess_text(self, text):
        text = ' '.join(text.split())  # Hapus spasi berlebih
        text = text.lower()  # Ubah ke lowercase
        text = re.sub(r'[^a-zA-Z0-9\s.,!?]', '', text)  # Hapus karakter khusus
        return text

    def __getitem__(self, index):
        student_answer = self.preprocess_text(str(self.dataframe.iloc[index]['answer']))
        score = self.dataframe.iloc[index]['normalized_score']

        if self.use_reference and 'reference_answer' in self.dataframe.columns:
            reference_answer = self.preprocess_text(str(self.dataframe.iloc[index]['reference_answer']))
            encoding = self.tokenizer.encode_plus(
                reference_answer,
                student_answer,
                add_special_tokens=True,
                max_length=512,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
        else:
            encoding = self.tokenizer.encode_plus(
                student_answer,
                add_special_tokens=True,
                max_length=512,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )

        encoding = {key: tensor.squeeze(0) for key, tensor in encoding.items()}
        encoding['labels'] = torch.tensor(score, dtype=torch.float)

        return encoding

    def get_max_length(self, index):
        """Menghitung panjang maksimum tokenized input berdasarkan apakah reference_answer digunakan atau tidak."""
        student_answer = str(self.dataframe.iloc[index]['answer'])

        if self.use_reference and 'reference_answer' in self.dataframe.columns:
            reference_answer = str(self.dataframe.iloc[index]['reference_answer'])
            encoding = self.tokenizer.encode_plus(
                reference_answer,
                student_answer,
                add_special_tokens=True,
                return_tensors='pt'
            )
        else:
            encoding = self.tokenizer.encode_plus(
                student_answer,
                add_special_tokens=True,
                return_tensors='pt'
            )

        return encoding['input_ids'].flatten().shape[0]

In [3]:
import torch
import torch.nn as nn
from transformers import AutoModel, AlbertConfig, AlbertModel

class RegressionModel(nn.Module):
    def __init__(self, model_name='indobenchmark/indobert-lite-base-p2', dropout=0.1):
        super().__init__()
        # load pretrained model
        self.config_model = AlbertConfig.from_pretrained(model_name)
        self.config_model.intermediate_size = 2048
        self.model = AlbertModel(self.config_model)

        self.dropout = nn.Dropout(p=dropout, inplace=False)
        self.regression_layer = nn.Linear(self.model.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        x = self.dropout(cls_embedding)
        score = self.regression_layer(x)
        return score

In [4]:
class inference:
    def __init__(self, df, model_name, model_dir, MODEL_CLASS):
        self.model = MODEL_CLASS(model_name).to(device)
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        if(model_dir != ""):
            self.model.load_state_dict(torch.load(model_dir, weights_only=True))
        self.dataset = AutomaticScoringDataset(df, self.tokenizer, use_reference=True)

    def get_prediction(self, index):
        self.model.eval()
        data = self.dataset[index]
        label = round(data['labels'].item(),2)
        data_cuda = {k: v.to(device) for k, v in data.items()}
        input_ids = data_cuda['input_ids'].unsqueeze(0)
        attention_mask = data_cuda['attention_mask'].unsqueeze(0)
        token_type_ids = data_cuda['token_type_ids'].unsqueeze(0)
        with torch.no_grad():
            predictions = self.model(input_ids, attention_mask, token_type_ids)

        predicted_score = round(predictions.squeeze().item(), 2)
        return label, predicted_score

In [5]:
df = pd.read_csv("../../../data/aes_dataset_indo_unseen.csv")
print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 219 entries, 0 to 218
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   question          219 non-null    object 
 1   reference_answer  219 non-null    object 
 2   answer            219 non-null    object 
 3   score             219 non-null    float64
 4   normalized_score  219 non-null    float64
 5   dataset           219 non-null    object 
 6   dataset_num       219 non-null    object 
dtypes: float64(2), object(5)
memory usage: 12.1+ KB
None


,question,reference_answer,answer,score,normalized_score,dataset,dataset_num
0,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,"sumber tenaga, pemanis alami, menjaga sistem i...",27.0,0.27,analisis_essay,analisis_essay-1
1,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,"sebagai sumber energi, pemanis alami, menjaga ...",21.0,0.21,analisis_essay,analisis_essay-1
2,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,1. Sebagai energi. 2. Sebagai memperlancaar pe...,42.0,0.42,analisis_essay,analisis_essay-1
3,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,"untuk membuat kenyang, agar tidak lapar, agar ...",18.0,0.18,analisis_essay,analisis_essay-1
4,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat mempunyai peran penting untuk pros...,82.0,0.82,analisis_essay,analisis_essay-1


In [6]:
transformers.utils.logging.set_verbosity_error()
model_name = "indobenchmark/indobert-lite-base-p2"
model_dir = "../../../experiments/models/best_model/indobert_reduce2.pt"
pipeline = inference(df, model_name, model_dir, RegressionModel)
pretrained_pipeline = inference(df, model_name, model_dir="", MODEL_CLASS=RegressionModel)

In [7]:
df['bert_predicted'] = df.apply(lambda x: pipeline.get_prediction(x.name)[1], axis=1)
df['pre_bert_predicted'] = df.apply(lambda x: pretrained_pipeline.get_prediction(x.name)[1], axis=1)

c:\Users\User\Documents\Code\env\lib\site-packages\transformers\models\albert\modeling_albert.py:404: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attention_output = torch.nn.functional.scaled_dot_product_attention(


In [8]:
# 1_1
results = []

for col in df.columns[7:]:
    mse = mean_squared_error(df['normalized_score'], df[col])
    rmse = np.sqrt(mse)
    pearson_corr, _ = pearsonr(df['normalized_score'], df[col])
    
    # Append results as a dictionary
    results.append({'Predicted': col, 'dataset': 'indo unseen', 'MSE': mse, 'RMSE': rmse, 'Pearson Correlation': pearson_corr})

results_df = pd.DataFrame(results).set_index('Predicted')
results_df.head(10)

,dataset,MSE,RMSE,Pearson Correlation
Predicted,,,,
bert_predicted,indo unseen,0.028684,0.169364,0.581781
pre_bert_predicted,indo unseen,0.385414,0.620817,0.193096


## SYNTHETIC DATA

In [9]:
data = {
    "reference_answer" : [
        "Fungsi karbohidrat adalah sebagai pemasok energi, dapat memperlancar proses pada pencernaan, memberikan efek kenyang dengan kandungan selulosa-nya dan penyeimbang asam dan basa dalam tubuh"
    ]*12,
    "answer": [
        "Karbohidrat adalah sumber energi utama tubuh.",
        "Fungsi karbohidrat adalah sebagai sumber energi dan membantu memperlancar pencernaan.",
        "Karbohidrat memberikan energi untuk aktivitas harian serta memperlancar proses pencernaan.",
        "Selain sebagai sumber energi, karbohidrat juga membantu menjaga keseimbangan asam-basa dan memberikan rasa kenyang melalui seratnya.",
        "Karbohidrat berperan sebagai sumber energi, memperlancar pencernaan, memberikan rasa kenyang berkat seratnya, dan menyeimbangkan asam dan basa dalam tubuh.",
        "Karbohidrat menyediakan glukosa yang mendukung metabolisme tubuh dan memastikan proses pencernaan berjalan lancar.",
        "Karbohidrat memberikan tenaga, membantu pencernaan, dan menjaga kestabilan pH tubuh sebagai penyeimbang asam-basa.",
        "Fungsi karbohidrat meliputi penyediaan energi, dukungan pada metabolisme dan pencernaan, serta memberikan rasa kenyang dan menjaga keseimbangan asam-basa.",
        "Sebagai sumber energi utama, karbohidrat membantu pencernaan dan menciptakan rasa kenyang karena kandungan serat, serta berperan dalam menjaga keseimbangan pH tubuh.",
        "Karbohidrat berfungsi ganda: menyediakan energi yang dibutuhkan untuk aktivitas sehari-hari dan mendukung proses pencernaan; selain itu, melalui serat yang terkandung, mereka membantu menciptakan rasa kenyang serta mengatur keseimbangan asam dan basa dalam tubuh.",
        "memberikan nutrisi",
        "untuk nutrisi tubuh"
    ],
    "normalized_score": [
        0.18,
        0.22,
        0.24,
        0.47,
        0.80,
        0.30,
        0.35,
        0.75,
        0.82,
        0.85,
        0.03,
        0.03
    ]
}

new_data = pd.DataFrame(data)
new_data.head()

,reference_answer,answer,normalized_score
0,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat adalah sumber energi utama tubuh.,0.18
1,Fungsi karbohidrat adalah sebagai pemasok ener...,Fungsi karbohidrat adalah sebagai sumber energ...,0.22
2,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat memberikan energi untuk aktivitas ...,0.24
3,Fungsi karbohidrat adalah sebagai pemasok ener...,"Selain sebagai sumber energi, karbohidrat juga...",0.47
4,Fungsi karbohidrat adalah sebagai pemasok ener...,"Karbohidrat berperan sebagai sumber energi, me...",0.80


In [10]:
transformers.utils.logging.set_verbosity_error()
model_name = "indobenchmark/indobert-lite-base-p2"
model_dir = "../../../experiments/models/best_model/indobert_reduce2.pt"
pipeline = inference(new_data, model_name, model_dir, RegressionModel)
pretrained_pipeline = inference(new_data, model_name, model_dir="", MODEL_CLASS=RegressionModel)

In [11]:
new_data['bert_predicted'] = new_data.apply(lambda x: pipeline.get_prediction(x.name)[1], axis=1)
new_data['pre_bert_predicted'] = new_data.apply(lambda x: pretrained_pipeline.get_prediction(x.name)[1], axis=1)

In [12]:
results = []

for col in new_data.columns[3:]:
    mse = mean_squared_error(new_data['normalized_score'], new_data[col])
    rmse = np.sqrt(mse)
    pearson_corr, _ = pearsonr(new_data['normalized_score'], new_data[col])
    
    # Append results as a dictionary
    results.append({'Predicted': col, 'dataset': 'indo synthetic', 'MSE': mse, 'RMSE': rmse, 'Pearson Correlation': pearson_corr})

results_df_synt = pd.DataFrame(results).set_index('Predicted')
results_df_synt.head(10)

,dataset,MSE,RMSE,Pearson Correlation
Predicted,,,,
bert_predicted,indo synthetic,0.07585,0.275409,0.579960
pre_bert_predicted,indo synthetic,0.05810,0.241039,0.948352


## RAHUMOTO + SAG

In [13]:
df_sag = pd.read_csv("../../../data/aes_dataset_unseen.csv")
print(df_sag.info())
df_sag.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 445 entries, 0 to 444
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   question          445 non-null    object 
 1   reference_answer  445 non-null    object 
 2   answer            445 non-null    object 
 3   score             445 non-null    float64
 4   normalized_score  445 non-null    float64
 5   dataset           445 non-null    object 
 6   dataset_num       445 non-null    object 
dtypes: float64(2), object(5)
memory usage: 24.5+ KB
None


,question,reference_answer,answer,score,normalized_score,dataset,dataset_num
0,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,"sumber tenaga, pemanis alami, menjaga sistem i...",27.0,0.27,analisis_essay,analisis_essay-1
1,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,"sebagai sumber energi, pemanis alami, menjaga ...",21.0,0.21,analisis_essay,analisis_essay-1
2,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,1. Sebagai energi. 2. Sebagai memperlancaar pe...,42.0,0.42,analisis_essay,analisis_essay-1
3,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,"untuk membuat kenyang, agar tidak lapar, agar ...",18.0,0.18,analisis_essay,analisis_essay-1
4,Jelaskan kegunaan karbohidrat untuk tubuh kita.,Fungsi karbohidrat adalah sebagai pemasok ener...,Karbohidrat mempunyai peran penting untuk pros...,82.0,0.82,analisis_essay,analisis_essay-1


In [14]:
transformers.utils.logging.set_verbosity_error()
model_name = "indobenchmark/indobert-lite-base-p2"
model_dir = "../../../experiments/models/best_model/indobert_reduce2.pt"
pipeline = inference(df_sag, model_name, model_dir, RegressionModel)
pretrained_pipeline = inference(df_sag, model_name, model_dir="", MODEL_CLASS=RegressionModel)

In [15]:
df_sag['bert_predicted'] = df_sag.apply(lambda x: pipeline.get_prediction(x.name)[1], axis=1)
df_sag['pre_bert_predicted'] = df_sag.apply(lambda x: pretrained_pipeline.get_prediction(x.name)[1], axis=1)

In [16]:
# 1_1
results = []

for col in df_sag.columns[7:]:
    mse = mean_squared_error(df_sag['normalized_score'], df_sag[col])
    rmse = np.sqrt(mse)
    pearson_corr, _ = pearsonr(df_sag['normalized_score'], df_sag[col])
    
    # Append results as a dictionary
    results.append({'Predicted': col, 'dataset': 'indo & sag unseen', 'MSE': mse, 'RMSE': rmse, 'Pearson Correlation': pearson_corr})

results_df_mix = pd.DataFrame(results).set_index('Predicted')
results_df_mix.head(10)

,dataset,MSE,RMSE,Pearson Correlation
Predicted,,,,
bert_predicted,indo & sag unseen,0.046125,0.214768,0.78377
pre_bert_predicted,indo & sag unseen,0.633376,0.795849,0.38717


## CONCAT

In [17]:
concat_df = pd.concat([results_df.head(1), results_df_synt.head(1), results_df_mix.head(1)], axis=0)
concat_df = concat_df.reset_index()
concat_df = concat_df.set_index('dataset')
concat_df.head()

,Predicted,MSE,RMSE,Pearson Correlation
dataset,,,,
indo unseen,bert_predicted,0.028684,0.169364,0.581781
indo synthetic,bert_predicted,0.075850,0.275409,0.579960
indo & sag unseen,bert_predicted,0.046125,0.214768,0.783770
